## Import necessary libraries

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Import weather data

In [16]:
# ── Load price data ──────────────────────────────────────────────────────
df_prices = pd.read_csv(
    "raw data/Energy_Day-ahead_prices_Hour.csv",
    sep=";",
    encoding="utf-8-sig",        # handles the BOM character at start of file
    na_values="-",               # SMARD uses "-" for missing values
)

# ── Keep only the necessary columns ─────────────────────────────────────────────
df_prices = df_prices[["Start date", "Germany/Luxembourg [€/MWh] Calculated resolutions"]]
df_prices.columns = ["date", "price_eur_mwh"]

# ── Parse the date column ──────────────────────────────────────────────────────
df_prices["date"] = pd.to_datetime(df_prices["date"], format="%b %d, %Y %I:%M %p")

# ── Set as index and sort ──────────────────────────────────────────────────────
df_prices = df_prices.set_index("date").sort_index()

print(df_prices.head())

                     price_eur_mwh
date                              
2019-01-01 00:00:00          28.32
2019-01-01 01:00:00          10.07
2019-01-01 02:00:00          -4.08
2019-01-01 03:00:00          -9.91
2019-01-01 04:00:00          -7.41


/var/folders/hk/j7q06lqj03s0hs7ptyd418h00000gn/T/ipykernel_68458/95861594.py:2: DtypeWarning: Columns (0: France [€/MWh] Calculated resolutions, 1: Slovenia [€/MWh] Calculated resolutions, 2: Hungary [€/MWh] Calculated resolutions) have mixed types. Specify dtype option on import or set low_memory=False.
  df_prices = pd.read_csv(


Sidenote: Electricity prices go *negative* when there is too much supply and not enough demand. Unlike gas or oil, electricity cannot easily be stored, so the grid must stay in perfect balance every second. When there's a surplus, someone has to pay to get rid of it.

## Import weather data

In [17]:
df_weather = pd.read_csv(
    "raw data/Weather_open-meteo.csv",
    skiprows=3,        # skip the metadata rows at the top
)
df_weather.columns = ["date", "temperature", "wind_speed", "radiation", "precipitation"]
df_weather["date"] = pd.to_datetime(df_weather["date"])
df_weather = df_weather.set_index("date").sort_index()

print(f"Weather loaded: {df_weather.shape}")

Weather loaded: (52608, 4)


In [18]:
print("PRICES:")
print(df_prices.head(3))
print(f"Index type: {df_prices.index.dtype}")
print(f"Shape: {df_prices.shape}")

print("\nWEATHER:")
print(df_weather.head(3))
print(f"Index type: {df_weather.index.dtype}")
print(f"Shape: {df_weather.shape}")

PRICES:
                     price_eur_mwh
date                              
2019-01-01 00:00:00          28.32
2019-01-01 01:00:00          10.07
2019-01-01 02:00:00          -4.08
Index type: datetime64[us]
Shape: (52608, 1)

WEATHER:
                     temperature  wind_speed  radiation  precipitation
date                                                                  
2019-01-01 00:00:00          6.4        13.5        0.0            0.0
2019-01-01 01:00:00          6.2        14.7        0.0            0.0
2019-01-01 02:00:00          6.2        15.3        0.0            0.0
Index type: datetime64[us]
Shape: (52608, 4)


Check if timestamps are aligned

In [19]:
print("First price timestamp: ", df_prices.index[0])
print("First weather timestamp:", df_weather.index[0])

print("\nLast price timestamp: ", df_prices.index[-1])
print("Last weather timestamp:", df_weather.index[-1])

First price timestamp:  2019-01-01 00:00:00
First weather timestamp: 2019-01-01 00:00:00

Last price timestamp:  2024-12-31 23:00:00
Last weather timestamp: 2024-12-31 23:00:00


In [20]:
df = df_prices.join(df_weather, how="inner")
print(f"Shape after merge: {df.shape}")
print(df.head(3))

Shape after merge: (52608, 5)
                     price_eur_mwh  temperature  wind_speed  radiation  \
date                                                                     
2019-01-01 00:00:00          28.32          6.4        13.5        0.0   
2019-01-01 01:00:00          10.07          6.2        14.7        0.0   
2019-01-01 02:00:00          -4.08          6.2        15.3        0.0   

                     precipitation  
date                                
2019-01-01 00:00:00            0.0  
2019-01-01 01:00:00            0.0  
2019-01-01 02:00:00            0.0  


In [21]:
print("Missing values per column:")
print(df.isna().sum())
print(f"\nTotal rows with any missing: {df.isna().any(axis=1).sum()}")

Missing values per column:
price_eur_mwh    0
temperature      0
wind_speed       0
radiation        0
precipitation    0
dtype: int64

Total rows with any missing: 0


## Derived Weather Features

##### `wind_power = wind_speed ** 3`
Wind turbine output is proportional to wind speed **cubed** — this is basic physics.
Wind speed of 10 km/h produces 8x less power than wind speed of 20 km/h, not just 2x less.
The cubed version is a much better predictor of how much wind electricity is being generated.

##### `solar_proxy = radiation.clip(lower=0)`
Radiation is already a good solar proxy.
`.clip(lower=0)` ensures there are no accidental negative values (defensive coding).

##### `heating_degree = (18 - temperature).clip(lower=0)`
When temperature drops below 18°C, people turn on heating → electricity demand goes up → price goes up.
The formula captures "how many degrees below 18°C is it right now".
- temperature = 5°C → heating_degree = 13
- temperature = 25°C → heating_degree = 0 (clips at 0, no heating needed)

##### `cooling_degree = (temperature - 24).clip(lower=0)`
Above 24°C people turn on air conditioning → demand goes up → price goes up.
- temperature = 30°C → cooling_degree = 6
- temperature = 15°C → cooling_degree = 0 (clips at 0, no cooling needed)

Instead of feeding raw temperature into the model, heating and cooling degree columns
directly encode **electricity demand driven by temperature** — which is what actually affects price.

## Why Derived Features Instead of Raw Data?

The model would still work with raw data, but derived features help for three reasons.

### 1. The relationship between temperature and price is not linear
Raw temperature has a **U-shaped** relationship with price — both very cold and very hot
temperatures push prices up, while mild temperatures (15–20°C) correspond to low demand.
A simple model cannot easily learn a U-shape from one column.
Splitting into `heating_degree` and `cooling_degree` makes both relationships linear and explicit.

### 2. Wind cubing is physics
With raw `wind_speed`, the model has to discover on its own that the relationship
with power generation is cubic. That requires more data and a more complex model.
`wind_speed ** 3` encodes known physics directly, giving the model a head start.

### 3. The model does not know domain knowledge
The model is just looking for correlations in numbers. It does not know that wind turbines
follow a power curve, or that humans turn on heating below 18°C.
Derived features encode that domain knowledge explicitly.

**Note:** Tree-based models (Random Forest, XGBoost) can learn non-linear relationships
on their own, so raw features would work reasonably well too.
The derived features matter more for linear models such as Linear Regression.
Keeping derived features demonstrates understanding of the domain.

In [22]:
df["wind_power"]     = df["wind_speed"] ** 3
df["solar_proxy"]    = df["radiation"].clip(lower=0)
df["heating_degree"] = (18 - df["temperature"]).clip(lower=0)
df["cooling_degree"] = (df["temperature"] - 24).clip(lower=0)

print(df[["temperature", "heating_degree", "cooling_degree", "wind_speed", "wind_power"]].head(10))

                     temperature  heating_degree  cooling_degree  wind_speed  \
date                                                                           
2019-01-01 00:00:00          6.4            11.6             0.0        13.5   
2019-01-01 01:00:00          6.2            11.8             0.0        14.7   
2019-01-01 02:00:00          6.2            11.8             0.0        15.3   
2019-01-01 03:00:00          6.6            11.4             0.0        16.6   
2019-01-01 04:00:00          6.4            11.6             0.0        17.2   
2019-01-01 05:00:00          6.2            11.8             0.0        21.6   
2019-01-01 06:00:00          5.8            12.2             0.0        23.3   
2019-01-01 07:00:00          5.5            12.5             0.0        24.8   
2019-01-01 08:00:00          5.3            12.7             0.0        22.1   
2019-01-01 09:00:00          5.2            12.8             0.0        22.7   

                     wind_power  
date 

Electricity consumption patterns are completely different on weekdays vs weekends.

Weekdays — factories, offices, and industrial plants are running full capacity. Demand is high, prices are higher.

Weekends — industry shuts down, offices are empty. Demand drops significantly, prices are lower.

This is one of the strongest patterns in electricity pricing. If your model doesn't know it's Saturday, it might predict a high price when in reality demand is low because nobody is working.

In [23]:
df["hour"]       = df.index.hour
df["weekday"]    = df.index.dayofweek
df["month"]      = df.index.month
df["is_weekend"] = (df["weekday"] >= 5).astype(int)

print(df[["hour", "weekday", "month", "is_weekend"]].head())

                     hour  weekday  month  is_weekend
date                                                 
2019-01-01 00:00:00     0        1      1           0
2019-01-01 01:00:00     1        1      1           0
2019-01-01 02:00:00     2        1      1           0
2019-01-01 03:00:00     3        1      1           0
2019-01-01 04:00:00     4        1      1           0


## Feature Engineering — Lag Features

Lag features add historical price information to help the model understand
the current price regime — something weather alone cannot capture.

### Why lag features?

Electricity prices have **momentum** — if prices were high yesterday,
they are likely high today. Weather features describe *why* prices move,
but lag features tell the model *where prices currently are*.

This is realistic for day-ahead forecasting: when predicting tomorrow's prices,
yesterday's prices are always already known.

### `price_lag_24` — same hour yesterday
Captures short-term price momentum.
If prices were elevated yesterday at 3pm, they are likely elevated today at 3pm.

### `price_lag_168` — same hour last week (7 days × 24 hours = 168 hours)
Captures the weekly demand cycle.
Electricity consumption follows a strong weekly pattern —
Monday 8am looks like last Monday 8am in terms of industrial and office demand.
Comparing the same hour on the same day of the week makes this a very strong predictor.

### Why drop the first 168 rows?
`price_lag_168` requires 7 days of history before it can produce a value.
The first 168 rows (2019-01-01 00:00 → 2019-01-07 23:00) have no "last week" to look back at
and are dropped with `dropna()`.
This removes 0.3% of the data — completely negligible over 6 years.

In [24]:
# Lag features
df["price_lag_24"]  = df["price_eur_mwh"].shift(24)   # same hour yesterday
df["price_lag_168"] = df["price_eur_mwh"].shift(168)  # same hour last week

# Drop rows where lag is NaN (first 168 hours have no lag history)
df = df.dropna()

print(f"Shape after adding lags: {df.shape}")
print(f"NaN values: {df.isna().sum().sum()}")

Shape after adding lags: (52440, 15)
NaN values: 0


In [25]:
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

Shape: (52440, 15)
Columns: ['price_eur_mwh', 'temperature', 'wind_speed', 'radiation', 'precipitation', 'wind_power', 'solar_proxy', 'heating_degree', 'cooling_degree', 'hour', 'weekday', 'month', 'is_weekend', 'price_lag_24', 'price_lag_168']

First few rows:


,price_eur_mwh,temperature,wind_speed,radiation,precipitation,wind_power,solar_proxy,heating_degree,cooling_degree,hour,weekday,month,is_weekend,price_lag_24,price_lag_168
date,,,,,,,,,,,,,,,
2019-01-08 00:00:00,17.94,4.2,22.9,0.0,0.6,12008.989,0.0,13.8,0.0,0,1,1,0,46.03,28.32
2019-01-08 01:00:00,20.91,4.5,23.2,0.0,0.5,12487.168,0.0,13.5,0.0,1,1,1,0,47.98,10.07
2019-01-08 02:00:00,7.78,4.8,22.2,0.0,0.6,10941.048,0.0,13.2,0.0,2,1,1,0,47.84,-4.08
2019-01-08 03:00:00,14.33,5.5,19.7,0.0,1.3,7645.373,0.0,12.5,0.0,3,1,1,0,46.11,-9.91
2019-01-08 04:00:00,18.56,5.6,21.4,0.0,2.6,9800.344,0.0,12.4,0.0,4,1,1,0,46.08,-7.41


With time series data never split randomly: the model would train on data from 2023 and test on 2019 -> it would be learning from the future to predict the past. That's called "data leakage" and makes results meaningless.

I trained on 4 years including the energy crisis onset, and tested on the post-crisis recovery period. This tests whether the model generalizes to a new price regime it was not explicitly trained on — a much harder and more realistic evaluation than random splitting.

In [26]:
df.to_csv("df_final.csv")
print("Saved!")

Saved!
